In [ ]:
import sys
from pathlib import Path

# Install libraries used in this notebook
!{sys.executable} -m pip install -q numpy pandas scikit-learn torch xgboost tabpfn joblib

# Resolve the repository root in a portable way
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "data" / "input").exists():
        ROOT_DIR = candidate.resolve()
        break
else:
    ROOT_DIR = Path(r"e:/LYQ/work/250916remote_model").resolve()

DATA_DIR = ROOT_DIR / "data" / "input"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT_DIR)
print("Data directory:", DATA_DIR)


# UAV-Landsat-8 SRF

## UAV data to Landsat-8 SRF data

In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

df = pd.read_csv(
    DATA_DIR / "UAV.csv",
    encoding="gbk"
)

# UAV光谱
uav = df.iloc[:, :164]

# 属性列
other_cols = df.iloc[:, 164:]

# 波长
uav_wl = uav.columns.astype(float).values

# 光谱矩阵
uav_spec = uav.values

#读取 Landsat-8 SRF
def read_l8_srf(fname):

    data = pd.read_csv(
        fname,
        skiprows=4,
        sep=r"\s+",
        header=None,
        names=["wn","response"]
    )

    # cm^-1 → nm
    wl = 1e7 / data["wn"]

    out = pd.DataFrame({
        "wavelength": wl,
        "response": data["response"]
    })

    out = out.sort_values("wavelength")

    return out

#读取各波段 SRF
srf_dir = ROOT_DIR / "SRF" / "rtcoef_landsat_8_oli_srf"

band_files = {
    "B1":"rtcoef_landsat_8_oli_srf_ch01.txt",
    "B2":"rtcoef_landsat_8_oli_srf_ch02.txt",
    "B3":"rtcoef_landsat_8_oli_srf_ch03.txt",
    "B4":"rtcoef_landsat_8_oli_srf_ch04.txt",
    "B5":"rtcoef_landsat_8_oli_srf_ch05.txt"
}

srf_dict = {}

for band,file in band_files.items():

    srf_dict[band] = read_l8_srf(
        srf_dir / file
    )

#Landsat-8卷积函数
def calc_band(
        spectrum,
        uav_wl,
        srf_wl,
        srf_rsp):

    f = interp1d(
        uav_wl,
        spectrum,
        kind="linear",
        bounds_error=False,
        fill_value=0
    )

    spec_interp = f(srf_wl)

    return np.trapz(
        spec_interp * srf_rsp,
        srf_wl
    ) / np.trapz(
        srf_rsp,
        srf_wl
    )

#批量转换 UAV → Landsat-8
l8_results = []

for spectrum in uav_spec:

    row = {}

    for band,srf in srf_dict.items():

        row[band] = calc_band(
            spectrum,
            uav_wl,
            srf["wavelength"].values,
            srf["response"].values
        )

    l8_results.append(row)

l8_df = pd.DataFrame(l8_results)

#拼接属性列并导出
result = pd.concat(
    [l8_df, other_cols],
    axis=1
)
out_file = DATA_DIR / "UAV_Landsat8_SRF.csv"

result.to_csv(
    out_file,
    index=False,
    encoding="utf-8-sig"
)

print("已保存：", out_file)

已保存： E:\LYQ\work\250916remote_model\data\input\UAV_Landsat8_SRF.csv


C:\Users\PC\AppData\Local\Temp\ipykernel_44944\3483515151.py:81: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(
C:\Users\PC\AppData\Local\Temp\ipykernel_44944\3483515151.py:84: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  ) / np.trapz(


## 训练

### CSF-FPE

In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import copy
import torch.nn.functional as F

SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
model_save_dir = os.path.join(DATA_DIR, "UAV-Landsat8SRF/CSF-FPE")
os.makedirs(model_save_dir, exist_ok=True)

#=========================
#⭐ Dataset selection
#=========================
selected_datasets = ["uav2l8"]

DATASET_CONFIG = {
    "uav": (0, "UAV.csv"),
    "sentinel": (1, "Sentinel-2.csv"),
    "landsat": (2, "Landsat-8.csv"),
    "uav2s2": (3, "UAV_Sentinel_S2A.csv"),
    "uav2l8": (4, "UAV_Landsat8_SRF.csv")
}

# =========================
# 0. Wavelength normalization
# =========================
GLOBAL_LAMBDA_MIN = 350.0
GLOBAL_LAMBDA_MAX = 1002

def normalize_wavelength(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

def normalize_delta(x):
    return (x - GLOBAL_LAMBDA_MIN) / (GLOBAL_LAMBDA_MAX - GLOBAL_LAMBDA_MIN)

# =========================
# ⭐ split loader
# =========================
def load_split(path, fold_id=0):
    with open(path, "r") as f:
        folds = json.load(f)

    split = folds[fold_id]
    return (
        np.array(split["train_idx"]),
        np.array(split["test_idx"])
    )

# =========================
# Dataset
# =========================
class SpectralDataset(Dataset):
    def __init__(self, file_path, dtype, x_mean, x_std):
        df = pd.read_csv(file_path, encoding='gbk')
        self.dtype = dtype

        temp = df.iloc[:, -2].values.astype(np.float32)
        self.x = df.iloc[:, :-2].values.astype(np.float32)
        self.y = df.iloc[:, -1].values.astype(np.float32)

        # NDWI filter
        if dtype == 1:
            ndwi = (df['B3'] - df['B8']) / (df['B3'] + df['B8'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        elif dtype == 2:
            ndwi = (df['SR_B3'] - df['SR_B5']) / (df['SR_B3'] + df['SR_B5'] + 1e-8)
            mask = ndwi > 0.1
            self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # clean
        mask = np.all(self.x >= 0, axis=1)
        self.x, self.y, temp = self.x[mask], self.y[mask], temp[mask]

        # global norm
        self.x = (self.x - x_mean) / (x_std + 1e-6)

        # wavelength
        if dtype == 0:
            self.lambda_ = np.arange(350, 1003, 4).astype(np.float32)
            self.delta = np.full_like(self.lambda_, 4.0)
            self.group_id = np.arange(len(self.x)) // 10

        elif dtype == 1 or dtype == 3:
            bands = [(443,20),(490,65),(560,35),(665,30),(705,15),
                     (740,15),(783,20),(842,115),(865,20),(945,20)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        elif dtype == 2 or dtype == 4:
            bands = [(443,20),(482,65),(561,75),(655,50),(865,40)]
            self.lambda_ = np.array([b[0] for b in bands], dtype=np.float32)
            self.delta = np.array([b[1] for b in bands], dtype=np.float32)

        self.lambda_ = normalize_wavelength(self.lambda_)
        self.delta = normalize_delta(self.delta)

        self.temp = (temp - temp.mean()) / (temp.std() + 1e-6)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return {
            "x": torch.tensor(self.x[idx]),
            "lambda": torch.tensor(self.lambda_),
            "delta": torch.tensor(self.delta),
            "y": torch.tensor(self.y[idx]),
            "type": torch.tensor(self.dtype),
            "temp": torch.tensor(self.temp[idx])
        }

# =========================
# Model
# =========================
class SpectralFusionModel(nn.Module):
    def __init__(self, d_model=64, grid_size=64):
        super().__init__()

        self.d_model = d_model

        # =========================
        # 1. Fixed spectral grid
        # =========================
        self.register_buffer(
            "grid",
            torch.linspace(0, 1, grid_size)
        )

        # =========================
        # 2. Fourier Encoding
        # =========================
        self.fourier_B = nn.Parameter(
            torch.logspace(0, 2, d_model // 2)
        )
        self.fourier_scale = nn.Parameter(torch.ones(1))

        # Feature fusion layer
        self.fusion = nn.Linear(d_model * 2, d_model)

        # =========================
        # 3. Input embedding
        # =========================
        self.input_embed = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        self.temp_embed = nn.Sequential(
            nn.Linear(1, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

        # =========================
        # 4. FiLM conditioning
        # =========================
        self.film = nn.Sequential(
            nn.Linear(4, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model),
            nn.SiLU(),
            nn.Linear(d_model, d_model * 2)
        )

        # =========================
        # 5. Transformer Encoder
        # =========================
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2
        )

        # =========================
        #  Regression head
        # =========================
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Linear(d_model//2, 1)
        )

    # =========================
    # Fourier Encoding
    # =========================
    def fourier_encoding(self, x):
        x = x.unsqueeze(-1)  # (B, N, 1)

        proj = x * self.fourier_B * self.fourier_scale

        return torch.cat([
            torch.sin(proj),
            torch.cos(proj)
        ], dim=-1)

    # =========================
    # spectral projection
    # =========================
    def spectral_project(self, z, wl, delta):

        grid = self.grid.to(z.device).unsqueeze(0).unsqueeze(0)

        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        sigma = delta + 1e-6

        weight = torch.exp(-((wl - grid) ** 2) / (sigma ** 2))
        weight = weight / (weight.sum(dim=1, keepdim=True) + 1e-8)

        return (z.unsqueeze(2) * weight.unsqueeze(-1)).sum(dim=1)

    # =========================
    # forward
    # =========================
    def forward(self, x, wl, delta, dtype, temp):

        # -------------------------
        # 1. input embedding
        # -------------------------
        x = x.unsqueeze(-1)
        wl = wl.unsqueeze(-1)
        delta = delta.unsqueeze(-1)

        inp = torch.cat([x, wl, delta], dim=-1)
        z = self.input_embed(inp)

        # -------------------------
        # 2. FiLM conditioning
        # -------------------------
        cond = torch.stack([
            dtype.float(),
            temp,
            wl.mean(dim=1).squeeze(-1),
            delta.mean(dim=1).squeeze(-1)
        ], dim=1)

        gamma, beta = self.film(cond).chunk(2, dim=1)
        z = gamma.unsqueeze(1) * z + beta.unsqueeze(1)

        # -------------------------
        # 3. spectral projection
        # -------------------------
        z = self.spectral_project(
            z,
            wl.squeeze(-1),
            delta.squeeze(-1)
        )

        # -------------------------
        # 4. Fourier positional encoding
        # -------------------------
        grid = self.grid.unsqueeze(0).expand(z.size(0), -1)

        pos = self.fourier_encoding(grid)

        z = torch.cat([z, pos], dim=-1)
        z = self.fusion(z)

        # -------------------------
        # 5. Transformer + CLS token pooling
        # -------------------------
        B = z.size(0)
        t = self.temp_embed(temp.unsqueeze(-1)).unsqueeze(1)
        z = torch.cat([t, z], dim=1)          # (B,N+1,d)

        z = self.encoder(z)

        z = z[:, 0]  # ⭐ CLS token output

        # -------------------------
        # 6. regression head
        # -------------------------
        return self.head(z).squeeze()

# =========================
# Loss
# =========================
def r2_aware_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)

    pred_c = pred - pred.mean()
    tgt_c = target - target.mean()

    corr = torch.sum(pred_c * tgt_c) / (
        torch.sqrt(torch.sum(pred_c ** 2)) *
        torch.sqrt(torch.sum(tgt_c ** 2)) + 1e-8
    )

    return mse + 0.5 * (1 - corr)

# =========================
# ⭐ evaluation
# =========================
def evaluate(model, loader, device):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch["x"].to(device),
                batch["lambda"].to(device),
                batch["delta"].to(device),
                batch["type"].to(device),
                batch["temp"].to(device)
            )

            y_true.extend(batch["y"].cpu().numpy())
            y_pred.extend(pred.cpu().numpy())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return mse, rmse, mae, r2


# =========================
# ⭐ train with early stopping
# =========================
def train():

    device = "cuda" if torch.cuda.is_available() else "cpu"

    paths = [
        (DATASET_CONFIG[name][0],
         os.path.join(DATA_DIR, DATASET_CONFIG[name][1]))
        for name in selected_datasets
    ]

    # =========================
    # global normalization
    # =========================
    all_x = []
    for _, path in paths:
        df = pd.read_csv(path, encoding='gbk')
        all_x.append(df.iloc[:, :-2].values.astype(np.float32))

    all_x = np.vstack(all_x)
    x_mean = all_x.mean(axis=0)
    x_std = all_x.std(axis=0) + 1e-6

    dataset = SpectralDataset(paths[0][1], paths[0][0], x_mean, x_std)

    # =========================
    # ⭐ 5-fold split
    # =========================
    folds = json.load(open(SPLIT_PATH))

    results = []

    for fold_id in range(5):

        print(f"\n================ FOLD {fold_id} ================")

        train_idx = np.array(folds[fold_id]["train_idx"])
        test_idx  = np.array(folds[fold_id]["test_idx"])

        train_ds = Subset(dataset, train_idx)
        test_ds  = Subset(dataset, test_idx)

        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_ds, batch_size=16)

        model = SpectralFusionModel().to(device)
        opt = torch.optim.Adam(model.parameters(), lr=3e-4)

        # =========================
        # ⭐ early stopping
        # =========================
        best_loss = float("inf")
        patience = 50
        wait = 0
        best_model = None

        for epoch in range(500):

            model.train()
            total_loss = 0

            for batch in train_loader:
                pred = model(
                    batch["x"].to(device),
                    batch["lambda"].to(device),
                    batch["delta"].to(device),
                    batch["type"].to(device),
                    batch["temp"].to(device)
                )

                loss = r2_aware_loss(pred, batch["y"].to(device))

                opt.zero_grad()
                loss.backward()
                opt.step()

                total_loss += loss.item()

            # =========================
            # ⭐ validation = test here
            # =========================
            mse, rmse, mae, r2 = evaluate(model, test_loader, device)

            print(
                f"Epoch {epoch:03d} | "
                f"TrainLoss {total_loss:.4f} | "
                f"Test RMSE {rmse:.4f} R2 {r2:.4f}"
            )

            # =========================
            # ⭐ save best model
            # =========================
            if rmse < best_loss:
                best_loss = rmse
                best_model = copy.deepcopy(model.state_dict())
                wait = 0

                torch.save(
                    best_model,
                    os.path.join(model_save_dir, f"best_model_fold{fold_id}.pth")
                )
            else:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered")
                    break

        # =========================
        # ⭐ final evaluation
        # =========================
        model.load_state_dict(best_model)

        mse, rmse, mae, r2 = evaluate(model, test_loader, device)

        print(f"\n[FOLD {fold_id} FINAL]")
        print(f"MSE: {mse:.4f}")
        print(f"RMSE: {rmse:.4f}")
        print(f"MAE: {mae:.4f}")
        print(f"R2: {r2:.4f}")

        results.append([mse, rmse, mae, r2])

    # =========================
    # ⭐ overall result
    # =========================
    results = np.array(results)

    print("\n================ FINAL 5-FOLD RESULT ================")
    print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
    print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
    print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
    print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

if __name__ == "__main__":
    train()

KeyboardInterrupt: 

### TabPFN

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
from tabpfn import TabPFNRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
# Suppress warnings
warnings.filterwarnings("ignore")
# =========================
# path
# =========================

SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")

csv_path = os.path.join(DATA_DIR, "UAV_Landsat8_SRF.csv")

# Output directory for predictions
save_dir = os.path.join(DATA_DIR, "UAV-Landsat8SRF/TabPFN")
os.makedirs(save_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# Load cross-validation splits
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# Evaluation metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# data loading
# =========================
raw = pd.read_csv(csv_path, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = min(6, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print("samples:", len(y), "features:", X.shape[1])
print("group samples:", num_samples)

# =========================
# load folds
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# result container
# =========================
results = []

# =========================
# 5-fold training 
# =========================
for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    # ===== 加标准化 =====
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # =========================
    # model
    # =========================
    model = TabPFNRegressor(
        n_estimators=32,
        device=device
    )

    # =========================
    # train
    # =========================
    model.fit(X_train, y_train)

    # =========================
    # predict
    # =========================
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    # =========================
    # evaluate
    # =========================
    train_metrics = evaluate(y_train, pred_train)
    test_metrics = evaluate(y_test, pred_test)

    print(f"[Train] MSE={train_metrics[0]:.4f} RMSE={train_metrics[1]:.4f} MAE={train_metrics[2]:.4f} R2={train_metrics[3]:.4f}")
    print(f"[Test ] MSE={test_metrics[0]:.4f}  RMSE={test_metrics[1]:.4f}  MAE={test_metrics[2]:.4f} R2={test_metrics[3]:.4f}")

    # =========================
    # save fold result
    # =========================
    results.append(test_metrics)

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred_test)

# =========================
# final summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
samples: 570 features: 6
group samples: 57

================ FOLD 0 ================
[Train] MSE=0.0028 RMSE=0.0530 MAE=0.0064 R2=0.9992
[Test ] MSE=1.0847  RMSE=1.0415  MAE=0.6511 R2=0.5835

================ FOLD 1 ================
[Train] MSE=0.0000 RMSE=0.0055 MAE=0.0041 R2=1.0000
[Test ] MSE=2.7321  RMSE=1.6529  MAE=1.0659 R2=0.1997

================ FOLD 2 ================
[Train] MSE=0.0003 RMSE=0.0172 MAE=0.0055 R2=0.9999
[Test ] MSE=0.4420  RMSE=0.6648  MAE=0.5357 R2=0.8190

================ FOLD 3 ================
[Train] MSE=0.0002 RMSE=0.0142 MAE=0.0054 R2=0.9999
[Test ] MSE=1.1092  RMSE=1.0532  MAE=0.8915 R2=0.6789

================ FOLD 4 ================
[Train] MSE=0.0000 RMSE=0.0048 MAE=0.0038 R2=1.0000
[Test ] MSE=3.6353  RMSE=1.9066  MAE=1.3273 R2=0.1038

================ FINAL 5-FOLD RESULT ================
MSE  : 1.8006 ± 1.1891
RMSE : 1.2638 ± 0.4510
MAE  : 0.8943 ± 0.2845
R2   : 0.4770 ± 0.2776


### Transformer

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path
# =========================
DATA_DIR = r"data/input"
CSV_PATH = os.path.join(DATA_DIR, "UAV_Landsat8_SRF.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
save_dir = os.path.join(DATA_DIR, "UAV-Landsat8SRF/Transformer")
os.makedirs(save_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load dataset
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = min(6, raw.shape[1])
target_col = "DO(mg/L"

X_df = raw.iloc[:, :max_feat].copy()

if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print("Samples:", len(y), "Features:", X.shape[1])

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

print("Group samples:", num_samples)
# =========================
# load split
# =========================
def load_folds(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# Transformer baseline model
# =========================
class TransformerBaseline(nn.Module):
    def __init__(self, d_model=64, nhead=4, num_layers=2):
        super().__init__()

        self.embed = nn.Linear(1, d_model)

        self.pos = nn.Parameter(torch.randn(1, 200, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: (B, N)
        x = x.unsqueeze(-1)
        x = self.embed(x)

        N = x.size(1)
        x = x + self.pos[:, :N, :].to(x.device)

        x = self.encoder(x)

        x = x.mean(dim=1)

        return self.head(x).squeeze()
    
# =========================
# load folds
# =========================
folds = load_folds(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # tensor
    # =========================
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # =========================
    # model
    # =========================
    model = TransformerBaseline().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # =========================
    # training loop
    # =========================
    best_loss = float("inf")
    patience = 50
    wait = 0

    for epoch in range(500):

        model.train()

        pred = model(X_train)
        loss = loss_fn(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        # early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_model = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # =========================
    # evaluation
    # =========================
    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        pred = model(X_test).cpu().numpy()

    mse, rmse, mae, r2 = evaluate(y_test, pred)

    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    results.append([mse, rmse, mae, r2])

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred)
    torch.save(best_model, os.path.join(save_dir, f"fold_{fold_id}.pth"))

# =========================
# final result
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
Samples: 570 Features: 6
Group samples: 57

================ FOLD 0 ================
MSE=2.5639, RMSE=1.6012, MAE=1.1507, R2=0.0155

================ FOLD 1 ================
MSE=1.5628, RMSE=1.2501, MAE=0.7697, R2=0.5422

================ FOLD 2 ================
MSE=1.2185, RMSE=1.1039, MAE=0.8439, R2=0.5009

================ FOLD 3 ================
MSE=1.6022, RMSE=1.2658, MAE=0.9640, R2=0.5362

================ FOLD 4 ================
MSE=3.8218, RMSE=1.9550, MAE=1.4690, R2=0.0578

================ FINAL 5-FOLD RESULT ================
MSE  : 2.1539 ± 0.9464
RMSE : 1.4352 ± 0.3067
MAE  : 1.0395 ± 0.2504
R2   : 0.3305 ± 0.2407


### Residual MLP

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path
# =========================
DATA_DIR = r"data/input"
CSV_PATH = os.path.join(DATA_DIR, "UAV_Landsat8_SRF.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")
save_dir = os.path.join(DATA_DIR, "UAV-Landsat8SRF/Residual MLP")
os.makedirs(save_dir, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =========================
# load dataset
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")

raw.columns = raw.columns.astype(str).str.strip().str.replace('\ufeff', '', regex=False)

max_feat = min(6, raw.shape[1])
target_col = "DO(mg/L"

X_df = raw.iloc[:, :max_feat].copy()

if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values.astype(np.float32)
y = data.iloc[:, -1].values.astype(np.float32)

print("Samples:", len(y), "Features:", X.shape[1])

# =========================
# group
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

print("Group samples:", num_samples)

# =========================
# load folds
# =========================
def load_folds(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# metrics
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# MLP model
# =========================
class ResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)
        self.relu = nn.ReLU()

        # Project feature dimensions to a common space if mismatched
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        return self.relu(self.fc(x) + self.proj(x))


class MLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()

        self.block1 = ResidualBlock(in_dim, 256)
        self.block2 = ResidualBlock(256, 128)
        self.block3 = ResidualBlock(128, 64)
        self.block4 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        return x.squeeze()

# =========================
# folds
# =========================
folds = load_folds(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # tensor
    # =========================
    X_train = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)

    # =========================
    # model
    # =========================
    model = MLP(X.shape[1]).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    # =========================
    # training loop
    # =========================
    best_loss = float("inf")
    patience = 50
    wait = 0

    for epoch in range(500):

        model.train()

        pred = model(X_train)
        loss = loss_fn(pred, y_train)

        opt.zero_grad()
        loss.backward()
        opt.step()

        # early stopping
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_model = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # =========================
    # evaluation
    # =========================
    model.load_state_dict(best_model)

    model.eval()
    with torch.no_grad():
        pred = model(X_test).cpu().numpy()

    mse, rmse, mae, r2 = evaluate(y_test, pred)

    print(f"MSE={mse:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")

    results.append([mse, rmse, mae, r2])

    np.save(os.path.join(save_dir, f"fold_{fold_id}_pred.npy"), pred)
    torch.save(best_model, os.path.join(save_dir, f"fold_{fold_id}.pth"))

# =========================
# final result
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE  : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE : {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE  : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2   : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Device: cuda
Samples: 570 Features: 6
Group samples: 57

================ FOLD 0 ================
MSE=1.0227, RMSE=1.0113, MAE=0.6933, R2=0.6073

================ FOLD 1 ================
MSE=1.4706, RMSE=1.2127, MAE=0.8840, R2=0.5692

================ FOLD 2 ================
MSE=1.1753, RMSE=1.0841, MAE=0.7464, R2=0.5186

================ FOLD 3 ================
MSE=3.6308, RMSE=1.9055, MAE=1.3027, R2=-0.0511

================ FOLD 4 ================
MSE=3.9507, RMSE=1.9876, MAE=1.3810, R2=0.0260

================ FINAL 5-FOLD RESULT ================
MSE  : 2.2500 ± 1.2703
RMSE : 1.4402 ± 0.4192
MAE  : 1.0015 ± 0.2859
R2   : 0.3340 ± 0.2854


### XGBoost

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import copy

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold
import xgboost as xgb

# =========================
# path config
# =========================
DATA_DIR = r"\data\input"
CSV_PATH = os.path.join(DATA_DIR, "UAV_Landsat8_SRF.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")

SAVE_DIR = os.path.join(DATA_DIR, "UAV-Landsat8SRF/XGBoost")
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# load data
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip()

# feature selection
max_feat = min(6, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

if target_col not in raw.columns:
    raise ValueError(f"Target column {target_col} not found")

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print(f"Samples: {len(y)}, Features: {X.shape[1]}")

# =========================
# group definition (same as PI-Transformer)
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print(f"Grouped samples: {num_samples}")

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# evaluation
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# load folds
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # model
    # =========================
    model = xgb.XGBRegressor(
        n_estimators=2000,
        max_depth=1,
        learning_rate=0.01,
        subsample=0.7,
        colsample_bytree=0.7,
        reg_lambda=1.0,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=1234
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    # =========================
    # prediction
    # =========================
    y_pred = model.predict(X_test)

    mse, rmse, mae, r2 = evaluate(y_test, y_pred)

    print(f"Fold {fold_id} | MSE: {mse:.4f} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}")

    # =========================
    # save model
    # =========================
    model_path = os.path.join(SAVE_DIR, f"xgb_fold{fold_id}.json")
    model.save_model(model_path)

    print(f"Saved: {model_path}")

    results.append([mse, rmse, mae, r2])

# =========================
# summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE: {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2  : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Samples: 570, Features: 6
Grouped samples: 57

================ FOLD 0 ================
Fold 0 | MSE: 0.9585 RMSE: 0.9791 MAE: 0.6724 R2: 0.6320
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/XGBoost\xgb_fold0.json

================ FOLD 1 ================
Fold 1 | MSE: 1.0294 RMSE: 1.0146 MAE: 0.7996 R2: 0.6985
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/XGBoost\xgb_fold1.json

================ FOLD 2 ================
Fold 2 | MSE: 1.7398 RMSE: 1.3190 MAE: 0.8800 R2: 0.2874
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/XGBoost\xgb_fold2.json

================ FOLD 3 ================
Fold 3 | MSE: 1.8140 RMSE: 1.3469 MAE: 1.0253 R2: 0.4748
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/XGBoost\xgb_fold3.json

================ FOLD 4 ================
Fold 4 | MSE: 2.4277 RMSE: 1.5581 MAE: 1.1139 R2: 0.4015
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/XGBoost\xgb_fold4.json

================ F

### Random Forest

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import copy
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# =========================
# path config
# =========================
DATA_DIR = r"\data\input"
CSV_PATH = os.path.join(DATA_DIR, "UAV_Landsat8_SRF.csv")
SPLIT_PATH = os.path.join(DATA_DIR, "UAV/uav_splits.json")

SAVE_DIR = os.path.join(DATA_DIR, "UAV-Landsat8SRF/RandomForest")
os.makedirs(SAVE_DIR, exist_ok=True)

# =========================
# load data
# =========================
raw = pd.read_csv(CSV_PATH, encoding="gbk")
raw.columns = raw.columns.astype(str).str.strip()

# =========================
# feature / target
# =========================
max_feat = min(6, raw.shape[1])
features = raw.iloc[:, :max_feat].copy()

target_col = "DO(mg/L"

if target_col not in raw.columns:
    raise ValueError(f"Target column {target_col} not found")

X_df = features.copy()
if target_col in X_df.columns:
    X_df = X_df.drop(columns=[target_col])

X = X_df.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(raw[target_col], errors="coerce")

data = pd.concat([X, y], axis=1).dropna()

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

print(f"Samples: {len(y)}, Features: {X.shape[1]}")

# =========================
# group definition 
# =========================
group_size = 10
num_samples = len(y) // group_size

valid_len = num_samples * group_size
X = X[:valid_len]
y = y[:valid_len]

groups = np.repeat(np.arange(num_samples), group_size)

print(f"Grouped samples: {num_samples}")

# =========================
# load split
# =========================
def load_split(path):
    with open(path, "r") as f:
        return json.load(f)

# =========================
# evaluation
# =========================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, rmse, mae, r2

# =========================
# load splits
# =========================
folds = load_split(SPLIT_PATH)

# =========================
# training
# =========================
results = []

for fold_id in range(5):

    print(f"\n================ FOLD {fold_id} ================")

    train_idx = np.array(folds[fold_id]["train_idx"])
    test_idx = np.array(folds[fold_id]["test_idx"])

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # =========================
    # Random Forest model
    # =========================
    model = RandomForestRegressor(
        n_estimators=500,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    mse, rmse, mae, r2 = evaluate(y_test, y_pred)

    print(f"Fold {fold_id} | MSE: {mse:.4f} RMSE: {rmse:.4f} MAE: {mae:.4f} R2: {r2:.4f}")

    # =========================
    # save model
    # =========================
    model_path = os.path.join(SAVE_DIR, f"rf_fold{fold_id}.pkl")
    joblib.dump(model, model_path)

    print(f"Saved: {model_path}")

    results.append([mse, rmse, mae, r2])

# =========================
# summary
# =========================
results = np.array(results)

print("\n================ FINAL 5-FOLD RESULT ================")
print(f"MSE : {results[:,0].mean():.4f} ± {results[:,0].std():.4f}")
print(f"RMSE: {results[:,1].mean():.4f} ± {results[:,1].std():.4f}")
print(f"MAE : {results[:,2].mean():.4f} ± {results[:,2].std():.4f}")
print(f"R2  : {results[:,3].mean():.4f} ± {results[:,3].std():.4f}")

Samples: 570, Features: 6
Grouped samples: 57

================ FOLD 0 ================
Fold 0 | MSE: 0.7534 RMSE: 0.8680 MAE: 0.5752 R2: 0.7107
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/RandomForest\rf_fold0.pkl

================ FOLD 1 ================
Fold 1 | MSE: 1.4371 RMSE: 1.1988 MAE: 0.8028 R2: 0.5790
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/RandomForest\rf_fold1.pkl

================ FOLD 2 ================
Fold 2 | MSE: 1.6201 RMSE: 1.2728 MAE: 0.8134 R2: 0.3365
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/RandomForest\rf_fold2.pkl

================ FOLD 3 ================
Fold 3 | MSE: 1.6647 RMSE: 1.2902 MAE: 0.9699 R2: 0.5181
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/RandomForest\rf_fold3.pkl

================ FOLD 4 ================
Fold 4 | MSE: 3.3003 RMSE: 1.8167 MAE: 1.3264 R2: 0.1863
Saved: E:\LYQ\work\250916remote_model\data\input\UAV-Landsat8SRF/RandomForest\rf_fold4.pkl

===